# A revolução dos Bancos Digitais melhorou ou piorou a experiência do consumidor brasileiro?

## Hipótese
Com o enfraquecimento dos bancos físicos e a forte necessidade de haver mais comodidade e praticidade aos clientes, houve a corrida das fintechs e bancos digitais para oferecer serviços que antes exigiam presença em agência. Mas será que a transformação digital foi devidamente preparada para esse grande fluxo?

Minha hipótese é que as instituições priorizaram não ficar para trás na corrida digital, investindo menos em segurança e acessibilidade para o consumidor. 

**Período de Análise:** 2017 a 2025
**Fonte:** Banco Central do Brasil - Ranking de Reclamações

## Configuração Inicial

Importação das bibliotecas que serão utilizadas ao longo do projeto:  `pandas` para manipulação de dados, `matplotlib` e `seaborn` para eventuais visualizações exploratórias.

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Carregamento dos Dados

Os dados do Banco Central são disponibilizados em CSVs separados por trimestre e ano. Para consolidar 9 anos de dados (2017–2025) em dois DataFrames únicos, percorri cada pasta de ano e empilhei os arquivos de mesma categoria usando `pd.concat`.

**Observação:** o dataset do 2º trimestre de 2022 não foi disponibilizado pelo Banco Central, totalizando 35 trimestres ao invés de 36.

In [50]:
import os
import glob

# Listas para armazenar os dados de cada arquivo
lista_reclamacoes = []
lista_irregularidades = []

# Percorrer cada pasta de ano (2017 a 2025)
for ano in range(2017, 2026):
    pasta = f'dados/{ano}'
    
    # Verificar se a pasta existe
    if not os.path.exists(pasta):
        print(f'⚠️ Pasta {pasta} não encontrada')
        continue
    
    # Ler todos os CSVs da pasta
    for arquivo in os.listdir(pasta):
        if arquivo.endswith('.csv'):
            caminho = os.path.join(pasta, arquivo)
            
            if 'Reclamacoes' in arquivo or 'Reclamações' in arquivo:
                df = pd.read_csv(caminho, sep=';', encoding='latin1')
                lista_reclamacoes.append(df)
                
            elif 'Irregularidades' in arquivo:
                df = pd.read_csv(caminho, sep=';', encoding='latin1')
                lista_irregularidades.append(df)

# Empilhar tudo em um único DataFrame
df_reclamacoes = pd.concat(lista_reclamacoes, ignore_index=True)
df_irregularidades = pd.concat(lista_irregularidades, ignore_index=True)

print(f'✅ Reclamações: {df_reclamacoes.shape[0]} linhas x {df_reclamacoes.shape[1]} colunas')
print(f'✅ Irregularidades: {df_irregularidades.shape[0]} linhas x {df_irregularidades.shape[1]} colunas')
print(f'\nPeríodo: {df_reclamacoes["Ano"].min()} a {df_reclamacoes["Ano"].max()}')
print(f'Trimestres por ano:\n{df_reclamacoes.groupby("Ano")["Trimestre"].nunique()}')

✅ Reclamações: 4663 linhas x 23 colunas
✅ Irregularidades: 100601 linhas x 15 colunas

Período: 2017 a 2025
Trimestres por ano:
Ano
2017    4
2018    4
2019    4
2020    4
2021    4
2022    3
2023    4
2024    4
2025    4
Name: Trimestre, dtype: int64


## Exploração Inicial

Análise da estrutura inicial dos dois DataFrames para identificar colunas relevantes, tipos de dados e necessidades de limpeza.

In [51]:
# verificando informações gerais do DataFrame de reclamações
df_reclamacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4663 entries, 0 to 4662
Data columns (total 23 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   Ano                                                           4663 non-null   int64  
 1   Trimestre                                                     4663 non-null   object 
 2   Categoria                                                     4663 non-null   object 
 3   Tipo                                                          4663 non-null   object 
 4   CNPJ IF                                                       4663 non-null   object 
 5   Instituição financeira                                        4663 non-null   object 
 6   Índice                                                        4663 non-null   object 
 7   Quantidade de reclamações reguladas procedentes               3393 no

In [52]:
# verificando informações gerais do DataFrame de irregularidades
df_irregularidades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100601 entries, 0 to 100600
Data columns (total 15 columns):
 #   Column                                                        Non-Null Count   Dtype  
---  ------                                                        --------------   -----  
 0   Ano                                                           100601 non-null  int64  
 1   Trimestre                                                     100601 non-null  object 
 2   Categoria                                                     100601 non-null  object 
 3   Tipo                                                          100601 non-null  object 
 4   CNPJ IF                                                       100601 non-null  object 
 5   Instituição financeira                                        100601 non-null  object 
 6   Irregularidade                                                100598 non-null  object 
 7   Quantidade de reclamações reguladas procedentes         

# ETL - Limpeza e Preparação dos Dados

Ao analisar a estrutura dos dados, identifiquei **23 colunas** no DataFrame de reclamações e **15 colunas** no DataFrame de irregularidades. Após avaliar a relevância de cada uma para as perguntas da análise, realizei as seguintes seleções:

**DataFrame de Reclamações (df_reclamacoes):** selecionei 9 colunas essenciais e descartei as demais. Incluindo colunas vazias geradas por formatação do CSV (Unnamed), colunas com cobertura inferior a 30% do período analisado (reclamações respondidas), e detalhamentos de clientes (CCS, SCR e FGC separados) já representados por uma coluna consolidada.

**DataFrame de Irregularidades (df_irregularidades):** selecionei 6 colunas essenciais, mantendo apenas as colunas de ligação entre as tabelas (ano, trimestre, instituição), o tipo de irregularidade e as métricas de reclamações procedentes e totais. Colunas como CNPJ, Unnamed e extrapoladas foram descartadas por serem irrelevantes ou redundantes para a análise.

Além da seleção de colunas, foram realizadas correções de tipo de dado: conversão de texto para número nas colunas de índice e quantidade de clientes, tratando separadores decimais (vírgula para 
ponto) e separadores de milhar.

In [ ]:
# Selecionando apenas as colunas relevantes para análise na DataFrame de reclamações

colunas_reclamacoes = ['Ano', 'Trimestre', 'Categoria', 'Tipo', 'Instituição financeira', 'Índice', 'Quantidade de reclamações reguladas procedentes', 'Quantidade total de reclamações', 'Quantidade total de clientes \x96 CCS e SCR']

df_reclamacoes = df_reclamacoes[colunas_reclamacoes]

print(f'Colunas mantidas: {df_reclamacoes.shape[1]}')
print(f'Linhas: {df_reclamacoes.shape[0]}')

Colunas mantidas: 9
Linhas: 4663


In [54]:
df_reclamacoes.dtypes

Ano                                                  int64
Trimestre                                           object
Categoria                                           object
Tipo                                                object
Instituição financeira                              object
Índice                                              object
Quantidade de reclamações reguladas procedentes    float64
Quantidade total de reclamações                    float64
Quantidade total de clientes  CCS e SCR            object
dtype: object

In [55]:
# Corrigindo o tipo de Dado da coluna 'índice' de texto para número

df_reclamacoes['Índice'] = (
    df_reclamacoes['Índice']
    .str.strip()
    .replace('', float('nan'))
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float))

#Verificando o resultado da correção
print(df_reclamacoes['Índice'].dtype)

float64


In [56]:
# Corrigindo o tipo de Dado da coluna 'Quantidade total de clientes \x96 CCS e SCR' de texto para número 

col_clientes = 'Quantidade total de clientes \x96 CCS e SCR'

df_reclamacoes[col_clientes] = pd.to_numeric(
    df_reclamacoes[col_clientes]
    .astype(str)
    .str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False),
    errors='coerce'
)

print(f'Tipo: {df_reclamacoes[col_clientes].dtype}')

Tipo: float64


In [60]:
# Renomeando colunas para facilitar a análise

df_reclamacoes = df_reclamacoes.rename(columns= {
    'Instituição financeira': 'instituicao',
    'Índice': 'indice',
    'Quantidade de reclamações reguladas procedentes': 'reclamacoes_procedentes',
    'Quantidade total de reclamações': 'reclamacoes_total',
    'Quantidade total de clientes \x96 CCS e SCR': 'total_clientes'
    })

# Verificando colunas renomeadas
df_reclamacoes.columns.tolist()

['Ano',
 'Trimestre',
 'Categoria',
 'Tipo',
 'instituicao',
 'indice',
 'reclamacoes_procedentes',
 'reclamacoes_total',
 'total_clientes']

In [61]:
# Padronizando todos os nomes para minúsculo e sem acento

df_reclamacoes.columns = df_reclamacoes.columns.str.lower()

df_reclamacoes.columns.tolist()

['ano',
 'trimestre',
 'categoria',
 'tipo',
 'instituicao',
 'indice',
 'reclamacoes_procedentes',
 'reclamacoes_total',
 'total_clientes']

In [62]:
# Verificando o DataFrame após as correções e padronizações

df_reclamacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4663 entries, 0 to 4662
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ano                      4663 non-null   int64  
 1   trimestre                4663 non-null   object 
 2   categoria                4663 non-null   object 
 3   tipo                     4663 non-null   object 
 4   instituicao              4663 non-null   object 
 5   indice                   1416 non-null   float64
 6   reclamacoes_procedentes  3393 non-null   float64
 7   reclamacoes_total        3393 non-null   float64
 8   total_clientes           4647 non-null   float64
dtypes: float64(4), int64(1), object(4)
memory usage: 328.0+ KB


## Limpeza do DataFrame de Irregularidades

Aplicação do mesmo processo de limpeza no segundo DataFrame: seleção de colunas relevantes, renomeação e padronização.

In [63]:
# Verificando o DataFrame de irregularidas para proceder com a limpeza e padronização dos dados 

print(f'Shape: {df_irregularidades.shape}')
df_irregularidades.dtypes

Shape: (100601, 15)


Ano                                                               int64
Trimestre                                                        object
Categoria                                                        object
Tipo                                                             object
CNPJ IF                                                          object
Instituição financeira                                           object
Irregularidade                                                   object
Quantidade de reclamações reguladas procedentes                 float64
Quantidade de reclamações reguladas - outras                    float64
Quantidade de reclamações não reguladas                         float64
Quantidade total de reclamações                                 float64
Unnamed: 11                                                     float64
Quantidade de reclamações reguladas procedentes extrapoladas    float64
Unnamed: 8                                                      

In [64]:
# Selecionando apenas as colunas relevantes para análise na DataFrame de irregularidades

colunas_irreg = ['Ano','Trimestre','Instituição financeira','Irregularidade','Quantidade de reclamações reguladas procedentes','Quantidade total de reclamações']

df_irregularidades = df_irregularidades[colunas_irreg]

In [65]:
# Renomeando colunas para facilitar a análise

df_irregularidades = df_irregularidades.rename(columns={
    'Instituição financeira': 'instituicao',
    'Irregularidade': 'irregularidade',
    'Quantidade de reclamações reguladas procedentes': 'reclamacoes_procedentes',
    'Quantidade total de reclamações': 'reclamacoes_total'
    })

In [66]:
# Padronizando todos os nomes para minúsculo e sem acentos

df_irregularidades.columns = df_irregularidades.columns.str.lower()

print(f'shape: {df_irregularidades.shape}')
df_irregularidades.columns.tolist()

shape: (100601, 6)


['ano',
 'trimestre',
 'instituicao',
 'irregularidade',
 'reclamacoes_procedentes',
 'reclamacoes_total']

## Investigação de Valores Nulos

Análise dos valores nulos identificados nos DataFrames para entender sua origem e decidir o tratamento adequado.

In [67]:
# Verificando valores nulos nos dois DataFrames

print('=== reclamações ===')
print(df_reclamacoes.isnull().sum())

print(f'\n=== irregularidades ===')
print(df_irregularidades.isnull().sum())

=== reclamações ===
ano                           0
trimestre                     0
categoria                     0
tipo                          0
instituicao                   0
indice                     3247
reclamacoes_procedentes    1270
reclamacoes_total          1270
total_clientes               16
dtype: int64

=== irregularidades ===
ano                            0
trimestre                      0
instituicao                    0
irregularidade                 3
reclamacoes_procedentes    29495
reclamacoes_total          29495
dtype: int64


In [68]:
# Comparando o total de clientes entre quem tem e que não tem índice para verificar se há alguma relação entre a presença do índice e o número de clientes

print('Com índice:')
print(df_reclamacoes[df_reclamacoes['indice'].notna()]['total_clientes'].describe())

print('\nSem índice:')
print(df_reclamacoes[df_reclamacoes['indice'].isna()]['total_clientes'].describe())

Com índice:
count    1.416000e+03
mean     1.769791e+07
std      3.120211e+07
min      1.300000e+01
25%      1.190479e+06
50%      4.195851e+06
75%      1.223142e+07
max      1.580557e+08
Name: total_clientes, dtype: float64

Sem índice:
count    3.231000e+03
mean     5.159383e+05
std      1.430874e+06
min      0.000000e+00
25%      7.148500e+03
50%      5.672600e+04
75%      3.735165e+05
max      1.335652e+07
Name: total_clientes, dtype: float64


In [ ]:
# Verificando quando as principais fintechs aparecem com índice

fintechs = ['NUBANK', 'PICPAY', 'C6', 'INTER', 'NEON', 'ORIGINAL']

for fintech in fintechs:
    dados = df_reclamacoes[
        df_reclamacoes['instituicao'].str.contains(fintech, case=False, na=False)
        & df_reclamacoes['indice'].notna()
    ]
    if len(dados) > 0:
        anos = sorted(dados['ano'].unique())
        print(f'{fintech}: aparece com índice a partir de {anos[0]} — {len(dados)} registros')
    else:
        print(f'{fintech}: nunca aparece com índice calculado')

NUBANK: aparece com índice a partir de 2019 — 19 registros
PICPAY: aparece com índice a partir de 2024 — 5 registros
C6: aparece com índice a partir de 2019 — 25 registros
INTER: aparece com índice a partir de 2017 — 34 registros
NEON: aparece com índice a partir de 2021 — 16 registros
ORIGINAL: aparece com índice a partir de 2018 — 29 registros


## Tratamento de Valores Nulos

Ao verificar os valores nulos nos dois DataFrames, um dado chamou atenção: a coluna **índice** no DataFrame de reclamações apresentava 3.247 valores nulos em 4.663 linhas — ou seja, **70% dos registros**.

O índice é a métrica principal do ranking do Banco Central: ele representa a quantidade de reclamações procedentes a cada um milhão de clientes da instituição. Ao investigar, identifiquei que os 
valores nulos correspondem a instituições de menor porte, cuja base de clientes é insuficiente para o cálculo, a mediana de clientes dessas instituições é de apenas 56 mil, contra 4,2 milhões nas que 
possuem índice.

**Decisão:** os nulos foram mantidos, pois não representam erro de dados, mas ausência intencional do indicador pelo Banco Central.

Esse achado revelou outro insight relevante: ao verificar as principais fintechs, identifiquei que elas foram progressivamente incorporadas ao cálculo do índice à medida que cresceram, o Inter já aparecia em 2017, o Nubank a partir de 2019, e o PicPay apenas em 2024. Essa cronologia por si só ilustra a evolução do setor digital bancário no Brasil.

## Resumo Final e Exportação dos Dados

Os dados estão limpos, padronizados e prontos para análise. Os dois DataFrames serão exportados em formato CSV para posterior carregamento no BigQuery, onde serão realizadas as análises em SQL.

In [70]:
# Resumo final dos dados limpos

print('=== df_reclamacoes ===')
print(f'Período: {df_reclamacoes["ano"].min()} a {df_reclamacoes["ano"].max()}')
print(f'Instituições únicas: {df_reclamacoes["instituicao"].nunique()}')
print(f'Linhas: {df_reclamacoes.shape[0]}')

print(f'\n=== df_irregularidades ===')
print(f'Período: {df_irregularidades["ano"].min()} a {df_irregularidades["ano"].max()}')
print(f'Tipos de irregularidade: {df_irregularidades["irregularidade"].nunique()}')
print(f'Linhas: {df_irregularidades.shape[0]}')

=== df_reclamacoes ===
Período: 2017 a 2025
Instituições únicas: 532
Linhas: 4663

=== df_irregularidades ===
Período: 2017 a 2025
Tipos de irregularidade: 260
Linhas: 100601


In [71]:
# Exportando os dados limpos para CSV

df_reclamacoes.to_csv('dados/reclamacoes_limpo.csv', index=False, encoding='utf-8')
df_irregularidades.to_csv('dados/irregularidades_limpo.csv', index=False, encoding='utf-8')

print('✅ Arquivos exportados com sucesso!')

✅ Arquivos exportados com sucesso!
